# ARGUS on Google Colab — bootstrap test

Smallest possible test that ARGUS can run inside a Colab notebook. Two cells: one for your LLM provider, one universal installer.

**Prerequisites:**
- A Google account with Colab access
- An LLM API key for one of the providers below (or any other provider with a TOML config you write yourself)
- The API key saved as a Colab Secret. Open the 🔑 sidebar → "Add new secret" → use the name shown in the template (`OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `ZAI_API_KEY`, ...) → toggle "Notebook access" on

**Provider templates:** see [argus_colab/README.md](https://github.com/klinucsd/sage/blob/main/argus_colab/README.md) for ready-to-paste blocks for OpenAI, Anthropic, ZAI, OpenRouter, NRP, and how to adapt one for any other OpenAI-compatible endpoint.

**Goal:** the final `%%ask` cell answers "what date is today?" using ARGUS's agent loop. If that works, the bootstrap pattern is sound and you can extend with MCP, skills, etc.

In [ ]:
# Cell 1 — declare your LLM (Python dict)
# (swap for Anthropic / Gemini / ZAI / NRP / etc. from argus_colab/README.md)
LLM = {
    "model": "gpt-4o-mini",
    "api_key_env": "OPENAI_API_KEY",
    # "url": "https://api.z.ai/api/coding/paas/v4",  # uncomment for ZAI / NRP / OpenRouter
    # "flavor": "anthropic",                          # or "gemini" — defaults to "openai"
}

In [ ]:
# Cell 2 — universal install (identical for every provider)
exec(__import__('urllib.request').urlopen(
    'https://raw.githubusercontent.com/klinucsd/sage/main/argus_colab/install.py'
).read().decode(), globals())

## Smoke test

If everything above succeeded, the `%%ask` cell magic is now registered. The next cell asks ARGUS a trivial question — it should answer using the agent loop (an LLM call + maybe one tool call) and render the result inline.

In [ ]:
%%ask
What date is today?

## If the smoke test works

Next steps to validate:
- Try `%%mcp` against a public MCP server (e.g. `https://wenokn.fastmcp.app/mcp`) to confirm tool-calling works
- Try `%%skill github.com/klinucsd/sage_skills/tree/main/...` to install a skill at runtime
- Mount Google Drive (`from google.colab import drive; drive.mount('/content/drive')`) for persistent state across sessions

## If the smoke test fails

Common issues:
- **API key not set** — re-run cell 3 after adding the Colab Secret. The error message names the exact env var your config.toml expects (e.g. `ZAI_API_KEY`).
- **Magic command not found** — re-run cell 4; if `%%ask` still isn't recognized, the magic registration in `sage_magic.py` failed (check the cell-4 output for tracebacks)
- **Import errors during cell 4** — Python version mismatch or missing dependency; check `!python --version` (need 3.11+) and `!pip show deepagents-code`